# Week 3, day 4 (morning) — Worksheet 03: Facts, dimensions, and the three kinds of measure

> *A fact table stores measurable business events... Dimension tables describe
> the "who, what, where, when, and how" around a fact.* — L03, slides 10 and 15

With the grain settled, every column in the model now has to go somewhere. Slide
16 gives the test: **measures and foreign keys** in the fact table, **attributes
and hierarchies** in the dimensions.

Sorting them is the easy half. The hard half is slide 11's table:

| Measure type | Meaning | Example |
|---|---|---|
| Additive | can be summed across all relevant dimensions at the defined grain | sales amount |
| Semi-additive | can be summed across some dimensions, but not all | account balance, inventory level |
| Non-additive | should not be summed directly | ratio, percentage, average discount rate |

A column that is safe to `SUM` and one that is not look identical in a table.
This worksheet makes each of the three behave, on real numbers, so you can tell
them apart by what they do rather than by what they are called.

**Question 10 is supposed to raise an error.**

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 03 — Facts, dimensions and measures. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr = load("enrollment")
tx = load("transaction")
dtype = load("discount_type")

# A first pass at enrollment-grain money, good enough for this sheet.
# Worksheet 07 builds the real one, with the missing-value rules applied.
per_enr = (tx.groupby("enrl_id")
             .agg(tuition_amount=("full_price", "max"),
                  amount_paid=("payment_amount", "sum"),
                  discount_type_id=("discount_type_id", "max"))
             .reset_index())
per_enr = per_enr.merge(
    dtype[["discount_type_id", "discount_amount"]],
    on="discount_type_id", how="left")
per_enr["discount_amount"] = per_enr["discount_amount"].fillna(0.0)
per_enr = per_enr.merge(enr[["enrl_id", "course_id", "cohort_id", "stu_id"]],
                        on="enrl_id", how="left")
per_enr["enrollment_count"] = 1

# Slide 13's sample daily product-store dataset, reproduced exactly.
inventory = pd.DataFrame({
    "date":    ["May 1, 2024"] * 4 + ["May 2, 2024"] * 4,
    "store":   ["Store A", "Store A", "Store B", "Store B"] * 2,
    "product": ["Product X", "Product Y"] * 4,
    "revenue": [10000, 5000, 20000, 8000, 12000, 6000, 18000, 7000],
    "profit":  [2000, 1000, 6000, 1600, 3000, 1200, 5400, 1400],
    "eod_inventory_units": [1200, 600, 800, 500, 1150, 550, 850, 450],
    "profit_margin_pct": [20.0, 20.0, 30.0, 20.0, 25.0, 20.0, 30.0, 20.0],
})

print("per_enr:  ", per_enr.shape)
print("inventory:", inventory.shape, "(slide 13's sample dataset)")

PART A — sorting the columns

### Question 1

Apply slide 16's test to the `enrollment` table. For each column print the name, its dtype, its distinct count, and your classification: `key`, `measure`, `attribute` or `degenerate`.
> **NOTE:** `enrl_id` identifies the fact row itself. It is not a foreign key to anything and it is not a measure — it is a *degenerate dimension*, and it stays in the fact table.

In [ ]:
############################
## Your Code Here
############################

### Question 2

Fact tables grow; dimension tables mostly do not. For each source table print its row count and the number of rows added in the last 90 days of `enrl_date`, using each table's own date column where it has one.
> **NOTE:** `growth over time` is the practical test for "is this a fact or a dimension". Tables with no date column at all are almost always dimensions.

In [ ]:
############################
## Your Code Here
############################

PART B — additive

### Question 3

Test `enrollment_count` and `amount_paid` for additivity: total them over the whole table, then re-total them after grouping by `course_id`, by `cohort_id`, and by both. Print all the totals.

In [ ]:
############################
## Your Code Here
############################

PART C — semi-additive

### Question 4

Use slide 13's sample dataset. Print it, then total `revenue` and `eod_inventory_units` per store per date, and then across both dates. Compare what each total means.
> **NOTE:** slide 13's own conclusion — 1,800 + 1,300 = 3,100 units is valid; 1,800 + 1,700 = 3,500 is not. Reproduce both.

In [ ]:
############################
## Your Code Here
############################

### Question 5

Show the right way to roll a semi-additive measure over time. For `eod_inventory_units` per store, print the sum, the average across dates, and the value on the latest date. Say which one you would put in a report.

In [ ]:
############################
## Your Code Here
############################

PART D — non-additive

### Question 6

Slide 25 asks *"which course-cohort groups have the highest discount rates?"*. Compute a discount rate per enrollment (`discount_amount / tuition_amount`), then produce a rate per **course-cohort group** two ways: the mean of the per-enrollment rates, and `SUM(discount_amount) / SUM(tuition_amount)`. Print both for the top five groups, and the two overall figures.
> **NOTE:** these are not two roundings of one number. They answer different questions.

In [ ]:
############################
## Your Code Here
############################

### Question 7

Look hard at the `n` column in question 6's ranking. Print the distribution of group sizes, how many groups have three enrollments or fewer, and the average of the 383 group rates against the true overall rate.
> **NOTE:** a rate is a ratio, and a ratio over a small denominator is mostly noise. Check what is behind the top of the ranking before reporting it.

In [ ]:
############################
## Your Code Here
############################

PART E — why attributes stay out of the fact table

### Question 8

Suppose `course_name` went into the fact table instead of `dim_course`. Build that column onto `per_enr` and print: the number of rows storing each name, the total characters stored, and the same figure for a dimension table holding each name once.

In [ ]:
############################
## Your Code Here
############################

### Question 9

Dimensions carry hierarchies. Build the `category -> program -> course` chain and print, for each level, the number of distinct values, then show the full path for three courses.
> **NOTE:** slide 15 calls these out — `region -> store`, `category -> product`. They are why a dimension can be drilled into.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Finally, treat a dimension attribute as a measure: run `enr_named["course_name"].mean()` after joining the names on. **This is supposed to fail.** Read the error and say what it is really objecting to.

In [ ]:
############################
## Your Code Here
############################